In [6]:
bootstrap_code = """#!/bin/bash
# Route all output to a log file we can inspect if needed
exec > >(tee /var/log/user-data.log|logger -t user-data -s 2>/dev/console) 2>&1

echo "--- STARTING CAPSTONE ML PIPELINE SETUP ---"

# 1. System Updates and Dependencies
dnf update -y
dnf install -y git python3 python3-pip wget tar

# 2. Install Python Libraries
pip3 install boto3 pandas scikit-learn numpy

# 3. Clone Repository
cd /home/ec2-user
git clone https://github.com/Iretha/IoT23-network-traffic-anomalies-classification.git
cd IoT23-network-traffic-anomalies-classification

# 4. Set up Data Directories
mkdir -p data/scenarios data/attacks experiments

# Note: The dataset download is placed here. 
# wget https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/iot23_combined.tar.gz
# tar -xzvf iot23_combined.tar.gz -C data/

# 5. Run the Machine Learning Experiment
echo "Starting Machine Learning run_experiments.py..."
python3 run_experiments.py

# 6. Budget Protection: Terminate the instance when finished
echo "Experiment complete. Shutting down instance to prevent further billing."
shutdown -h now
"""

with open("ml_bootstrap.sh", "w", encoding="utf-8") as f:
    f.write(bootstrap_code)

print("✅ Created ml_bootstrap.sh (EC2 Automation Script)")

✅ Created ml_bootstrap.sh (EC2 Automation Script)


In [5]:
terraform_code = """
provider "aws" {
  region = "us-east-1" # Change this if your AWS CLI is configured for a different region
}

# 1. IAM Role to allow EC2 to write to CloudWatch
resource "aws_iam_role" "ec2_cloudwatch_role" {
  name = "capstone_ec2_cloudwatch_role"
  assume_role_policy = jsonencode({
    Version = "2012-10-17"
    Statement = [{
      Action = "sts:AssumeRole"
      Effect = "Allow"
      Principal = { Service = "ec2.amazonaws.com" }
    }]
  })
}

# Attach the CloudWatch Logs policy to the role
resource "aws_iam_role_policy_attachment" "cloudwatch_logs_attach" {
  role       = aws_iam_role.ec2_cloudwatch_role.name
  policy_arn = "arn:aws:iam::aws:policy/CloudWatchLogsFullAccess"
}

resource "aws_iam_instance_profile" "ec2_profile" {
  name = "capstone_ec2_profile"
  role = aws_iam_role.ec2_cloudwatch_role.name
}

# 2. Security Group
resource "aws_security_group" "ml_sg" {
  name        = "capstone_ml_sg"
  description = "Allow outbound traffic for downloading dataset and repo"

  egress {
    from_port   = 0
    to_port     = 0
    protocol    = "-1"
    cidr_blocks = ["0.0.0.0/0"]
  }
}

# 3. The m5.xlarge ML Engine
resource "aws_instance" "ml_engine" {
  ami                  = "ami-0c101f26f147fa7fd" # Amazon Linux 2023 (us-east-1)
  instance_type        = "m5.xlarge"
  iam_instance_profile = aws_iam_instance_profile.ec2_profile.name
  security_groups      = [aws_security_group.ml_sg.name]
  
  # Pass the bash script we created to the instance
  user_data = file("${path.module}/ml_bootstrap.sh")

  tags = {
    Name = "Capstone-ML-Engine"
  }
}

# Output the Instance ID so you know it worked
output "ml_engine_instance_id" {
  value = aws_instance.ml_engine.id
}
"""

with open("main.tf", "w", encoding="utf-8") as f:
    f.write(terraform_code)

print("✅ Created main.tf (Terraform Configuration)")

✅ Created main.tf (Terraform Configuration)


In [4]:
import os
print(os.getcwd())

C:\Users\fordunt
